In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import sys
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from tqdm.auto import tqdm

sys.path.append(str(Path().resolve().parent / "src"))

from config import read_model_config, read_sampling_config
from data import create_gdphmm_dataset, StructureSelector
from models import build_model
from model_sampler import GaussianDiffusionSampler
from model_cache import DeepCache, CacheConfig
from utils.checkpointing import load_checkpoint
from mamba_model import remap_state_dict

In [4]:
# Configuration
MODEL_NAME = '201-beamdiff'
EPOCH = 199
SAMPLE_IDX = 0  # Which sample from the test set to use

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: mps


In [5]:
# Load model config and sampling config
model_config = read_model_config(MODEL_NAME)
sampling_config = read_sampling_config('ddpm-1000')

# Create conditioning selector
conditioning_groups = model_config['conditioning']['types']
conditioning_selector = StructureSelector(conditioning_groups)

# Load dataset
dataset = create_gdphmm_dataset(model_config, 'test', structure_groups=None)
print(f'Dataset size: {len(dataset)}')
print(f'Conditioning channels: {conditioning_selector.total_channels}')

FileNotFoundError: Preprocessed data not found at /Users/joseph.weibel@fhnw.ch/dev/dose-prediction-thesis/data/gdphmm-hn-vmat_slices/test. Run: python scripts/preprocess_gdphmm_slices.py --dataset gdphmm-hn-vmat --pad-size 256 256

In [ ]:
# Build and load model
model = build_model(model_config, conditioning_selector).to(device)

# Load checkpoint
checkpoint = load_checkpoint(MODEL_NAME, EPOCH)
model_state_dict = checkpoint['ema_model_state_dict']
if "t_proj_enc1.proj.weight" in model_state_dict:
    model_state_dict = remap_state_dict(model_state_dict)
model.load_state_dict(model_state_dict)
model.eval()

print(f'Loaded model from epoch {checkpoint["epoch"]}')

In [ ]:
# Custom sampler that captures intermediate steps
class GaussianDiffusionSamplerWithHistory(GaussianDiffusionSampler):
    """Modified DDPM sampler that stores intermediate denoising steps."""

    @torch.no_grad()
    def forward(self, x_T, condition, save_every=1):
        """
        Sample with history tracking.

        Args:
            x_T: Initial noise tensor
            condition: Conditioning information
            save_every: Save intermediate result every N steps (default: 1)

        Returns:
            Tuple of (final_result, history) where history is list of intermediate x values
        """
        x = x_T
        history = [x.detach().cpu().clone()]

        cache = DeepCache(self.cache_config)
        for i, t in enumerate(tqdm(self.sampling_indices, desc='Sampling')):
            cache.set_current_step(i)

            t_vec = torch.full((x.shape[0],), t, dtype=torch.long, device=x.device)
            model_output = self.model(x, condition, t_vec, cache)

            # Convert model output to x0 based on parameterization
            if self.parameterization == 'v':
                sqrt_alpha_bar = 1.0 / self.sqrt_recip_alphas_bar[i]
                sqrt_one_minus_alpha_bar = self.sqrt_one_minus_alphas_bar[i]
                x0_pred = sqrt_alpha_bar * x - sqrt_one_minus_alpha_bar * model_output
            else:
                eps = model_output
                x0_pred = self.sqrt_recip_alphas_bar[i] * (x - self.sqrt_one_minus_alphas_bar[i] * eps)

            # Compute mean using precomputed coefficients
            mean = self.mean_coef1[i] * x0_pred + self.mean_coef2[i] * x

            x = mean

            # Add noise (not on last step)
            if i < len(self.sampling_indices) - 1:
                noise = torch.randn_like(x)
                x += self.posterior_std[i] * noise

            # Save intermediate result
            if (i + 1) % save_every == 0 or i == len(self.sampling_indices) - 1:
                history.append(x.detach().cpu().clone())

        return x, history

In [ ]:
# Create sampler with history tracking
cache_config = CacheConfig(sampling_steps=sampling_config['T'], layers=[], intervals={}, warmup=0)
parameterization = model_config.get('diffusion', {}).get('parameterization', 'epsilon')

sampler = GaussianDiffusionSamplerWithHistory(
    model,
    model_config['diffusion']['beta_1'],
    model_config['diffusion']['beta_T'],
    model_config['diffusion']['T'],
    cache_config,
    sampling_config['T'],
    schedule=sampling_config['schedule'],
    parameterization=parameterization,
).to(device)

print(f'Created sampler with {len(sampler.sampling_indices)} steps')

In [ ]:
# Load a sample from the dataset
sample = dataset[SAMPLE_IDX]
structure = sample['structure']  # (C, H, W)
dose = sample['dose']  # (1, H, W)
file_name = sample['file_name']

print(f'Sample: {file_name}')
print(f'Structure shape: {structure.shape}')
print(f'Dose shape: {dose.shape}')

In [ ]:
# Prepare conditioning input
condition = conditioning_selector.build_structure(structure).unsqueeze(0).to(device)  # (1, C, H, W)
print(f'Condition shape: {condition.shape}')

# Generate initial noise
torch.manual_seed(42)  # For reproducibility
x_T = torch.randn(1, 1, structure.shape[1], structure.shape[2], device=device)
print(f'Initial noise shape: {x_T.shape}')

In [ ]:
# Run sampling with history (save every 10 steps for smoother animation)
SAVE_EVERY = 10  # Adjust for more/fewer frames

with torch.no_grad():
    final_pred, history = sampler(x_T, condition, save_every=SAVE_EVERY)

print(f'Captured {len(history)} frames')

In [ ]:
# Convert history to numpy arrays
frames = [h[0, 0].numpy() for h in history]  # Extract single channel

# Get the ground truth dose for reference
gt_dose = dose[0].numpy()

# Get CT slice for overlay
ct_slice = conditioning_selector.get_ct(structure)[0].numpy()
body_mask = dataset.selector.get_body(structure)[0].numpy()

print(f'Frame shape: {frames[0].shape}')
print(f'Value range - first frame: [{frames[0].min():.2f}, {frames[0].max():.2f}]')
print(f'Value range - last frame: [{frames[-1].min():.2f}, {frames[-1].max():.2f}]')
print(f'Value range - ground truth: [{gt_dose.min():.2f}, {gt_dose.max():.2f}]')

In [ ]:
# Create the animation - single panel showing denoising progress
fig, ax = plt.subplots(figsize=(6, 5))

# Determine color scale based on final prediction
vmin = 0
vmax = frames[-1].max()

# Initial frame
im = ax.imshow(np.clip(frames[0], 0, 1) * body_mask, cmap='jet', vmin=vmin, vmax=vmax)
ax.axis('off')

def update(frame_idx):
    """Update function for animation."""
    frame = frames[frame_idx]
    clipped_frame = np.clip(frame, 0, 1) * body_mask
    im.set_array(clipped_frame)
    return [im]

# Create animation
anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=100,  # 100ms between frames
    blit=True
)

plt.tight_layout()
plt.close()  # Don't show static figure
anim

In [ ]:
# Save as GIF
output_path = Path('diffusion_denoising.gif')
anim.save(output_path, writer='pillow', fps=10)
print(f'Animation saved to {output_path.absolute()}')

In [ ]:
# Display animation inline (alternative to the cell above)
HTML(anim.to_jshtml())

In [ ]:
# Static visualization of key timesteps
n_show = 8
indices = np.linspace(0, len(frames) - 1, n_show, dtype=int)

fig, axes = plt.subplots(1, n_show, figsize=(16, 2.5))

for i, idx in enumerate(indices):
    frame = frames[idx]
    clipped = np.clip(frame, 0, 1) * body_mask
    axes[i].imshow(clipped, cmap='jet', vmin=vmin, vmax=vmax)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('diffusion_keyframes.png', dpi=150, bbox_inches='tight')
plt.show()